# TinyDoc-VLM — Full 768 Retrain on Colab T4

Full-model (no LoRA) retrain at **768x768** with markdown-conversion synthetic data
(50K docs) + real benchmarks (OCRBench / FUNSD / CORD).

This is Tier-2 of the improvement plan (docs/model_improvements.md): after removing
the broken multi-task heads (A), raising resolution 384->768 (B), and adding the ngram
repetition penalty (C), this retrain is the "good margin" jump.

**Recommended method — Colab CLI (headless, best for long runs):**
Install the [google-colab-cli](https://github.com/googlecolab/google-colab-cli) (free;
uses your Colab account tier) and from your laptop run:

    pip install google-colab-cli && colab auth
    GPU=T4 STEPS=8000 ./training/run_colab_cli.sh

This provisions a T4 VM, runs `training/colab_train.py` headlessly, and tears it down.
Checkpoints land on Google Drive and `full_train.py` resumes from the latest `step_*`,
so free-tier 12h sessions accumulate progress across runs.

**This notebook** is the interactive alternative: Runtime -> T4 -> Restart, then Run all.
~30-40min data gen + training. Also fully resumable (guarded stages + step_* resume).

**Resumable end-to-end:** every stage is guarded — if its output already exists it is
skipped. The training stage auto-resumes from the latest `step_*` checkpoint, so if Colab
disconnects you just re-run and it continues (no lost progress). Make sure Drive is mounted
(Cell 1) so the checkpoints survive the restart.

## 1. Mount Google Drive (for checkpoint persistence)

In [ ]:
from google.colab import drive
import os

# Everything (repo, dataset, checkpoints) lives under WORK so it SURVIVES a
# Colab restart. If Drive mounts, WORK is on Drive; otherwise it is local
# (/content) and will be wiped on restart — re-run from scratch in that case.
try:
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/tinydoc-vlm'
    print('Google Drive mounted — work will persist across restarts.')
except Exception as e:
    print(f'Drive mount failed ({e}); using local dir (wiped on restart).')
    WORK = '/content/tinydoc-vlm'

os.makedirs(WORK, exist_ok=True)
print(f'WORK = {WORK}')

## 2. Clone repo & install deps

In [ ]:
import os, sys, subprocess, requests, zipfile, io, shutil

REPO_DIR = f'{WORK}/tinydoc-vlm'

def remove_dir(path):
    subprocess.run(['rm', '-rf', path], capture_output=True)

def preserve_data(repo):
    """Move generated data outside repo before wipe, return list of (src, dst) to restore."""
    saved = []
    for sub in ['data/training', 'checkpoints']:
        src = f'{repo}/{sub}'
        dst = f'{repo}_{sub.replace("/", "_")}_bak'
        if os.path.exists(src):
            shutil.move(src, dst)
            saved.append((sub, dst))
    return saved

def restore_data(repo, saved):
    for sub, bak_path in saved:
        dst = f'{repo}/{sub}'
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.move(bak_path, dst)

def zip_download(saved):
    print('Downloading repo via zip...')
    r = requests.get('https://github.com/eulogik/TinyDoc-VLM/archive/refs/heads/main.zip', timeout=60)
    z = zipfile.ZipFile(io.BytesIO(r.content))
    z.extractall(WORK)
    os.rename(f'{WORK}/TinyDoc-VLM-main', REPO_DIR)
    restore_data(REPO_DIR, saved)
    # Convert to real git repo so future updates use git pull (no re-download)
    subprocess.run(['git', 'init'], cwd=REPO_DIR, capture_output=True)
    subprocess.run(['git', 'remote', 'add', 'origin', 'https://github.com/eulogik/TinyDoc-VLM.git'],
                   cwd=REPO_DIR, capture_output=True)
    subprocess.run(['git', 'fetch', '--depth', '1', 'origin', 'main'], cwd=REPO_DIR, capture_output=True)
    subprocess.run(['git', 'checkout', '-B', 'main', 'origin/main'], cwd=REPO_DIR, capture_output=True)

saved = []
if not os.path.exists(f'{REPO_DIR}/data/synthetic/markdown_dataset.py'):
    saved = preserve_data(REPO_DIR)
    if os.path.exists(REPO_DIR):
        remove_dir(REPO_DIR)
    zip_download(saved)
elif os.path.exists(f'{REPO_DIR}/.git'):
    saved = preserve_data(REPO_DIR)
    subprocess.run(['git', 'pull'], cwd=REPO_DIR, capture_output=True)
    restore_data(REPO_DIR, saved)
    print('Existing repo updated via git pull')
else:
    saved = preserve_data(REPO_DIR)
    remove_dir(REPO_DIR)
    zip_download(saved)

assert os.path.exists(f'{REPO_DIR}/data/synthetic/markdown_dataset.py'), 'Download failed'
os.chdir(REPO_DIR)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'torchvision',
                '--index-url', 'https://download.pytorch.org/whl/cu124'], cwd=REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'sentencepiece', 'tokenizers', 'pillow', 'numpy',
                'pandas', 'tqdm', 'pyyaml', 'einops', 'faker', 'jinja2', 'pydantic',
                'datasets', 'accelerate'], cwd=REPO_DIR)
print('Repo ready — deps installed')

## 3. Generate 50K markdown-conversion dataset + real benchmarks

- data/synthetic/markdown_dataset.py renders synthetic docs -> prompt-target pairs
  (Convert the document to markdown:, Extract all text:, VQA, JSON).
- Real benchmarks are pulled from HuggingFace datasets into evaluation/data.
- data/build_training_dataset.py merges them into data/training/manifest.jsonl.

Lower NUM_DOCS (e.g. 15000) for a faster pilot. Generation runs on CPU (~1-2h for 50K).
If data/training/manifest.jsonl already exists with enough pairs, this stage is skipped.

In [ ]:
import os, sys, subprocess, shlex

REPO = f'{WORK}/tinydoc-vlm'
NUM_DOCS = 50000   # reduce to 15000 for a faster pilot
MANIFEST = 'data/training/manifest.jsonl'

def manifest_ok():
    if not os.path.exists(MANIFEST):
        return False
    n = sum(1 for _ in open(MANIFEST))
    return n >= 10000   # synthetic alone yields >10000 pairs for 50K docs

def run_streaming(cmd):
    print('$', shlex.join(cmd))
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')   # live progress, no buffering until the end
    p.wait()
    assert p.returncode == 0, f'Command failed (exit {p.returncode}): {shlex.join(cmd)}'

if manifest_ok():
    print(f'Manifest already exists ({sum(1 for _ in open(MANIFEST))} pairs) — skipping generation.')
else:
    # Only the 3 benchmarks actually used for training (OCRBench / FUNSD / CORD).
    # DocVQA (huge) and SROIE (no GT) are skipped by default.
    run_streaming([sys.executable, 'evaluation/download_benchmarks.py',
                   '--data-dir', 'evaluation/data',
                   '--benchmarks', 'ocrbench', 'funsd', 'cord'])
    print(f'Generating {NUM_DOCS} synthetic markdown docs + merging real benchmarks...')
    run_streaming([sys.executable, 'data/build_training_dataset.py',
                   '--num-docs', str(NUM_DOCS),
                   '--output-dir', 'data/training',
                   '--data-dir', 'evaluation/data'])
    assert manifest_ok(), 'Training data generation failed'

print(f'Combined training pairs: {sum(1 for _ in open(MANIFEST))}')

## 4. Initialize 768 model

training/init_768_model.py builds a 768 model from the 384 base checkpoint: copies the
trained decoder + compressor and interpolates the vision positional embeddings 384->768.
Skipped if checkpoints/init_768/config.json already exists.

In [ ]:
import os, sys, subprocess

REPO = f'{WORK}/tinydoc-vlm'
os.chdir(REPO)
if os.path.exists('checkpoints/init_768/config.json'):
    print('init_768 already exists — skipping.')
else:
    r = subprocess.run([sys.executable, 'training/init_768_model.py',
                        '--base', 'eulogik/TinyDoc-VLM-256M',
                        '--out', 'checkpoints/init_768'], cwd=REPO,
                       capture_output=True, text=True)
    print(r.stdout[-2500:])
    if r.returncode != 0:
        print('STDERR:', r.stderr[-2500:])
print('init_768 ready:', os.path.exists('checkpoints/init_768/config.json'))

## 5. Full fine-tune at 768 on T4

Trains ALL parameters (no LoRA) with bf16 + gradient checkpointing.
~0.5-1 step/s on T4 -> 30000 steps ~ 8-16h. Checkpoints saved locally; only the final
one is synced to Drive (intermediate checkpoints are throttled via --save-every to avoid
filling the disk). Skipped if the final checkpoint already exists.

## 5b. Auto-sync checkpoints to Hugging Face (background)

Starts a **daemon thread** that watches `checkpoints/full768/latest/` while the training cell below is busy.
Every ~25 min it converts the newest checkpoint to fp16 (on CPU, so it never touches VRAM used by training)
and uploads it to `eulogik/TinyDoc-VLM-768-checkpoints/latest`. It never overwrites a step higher than its own
already on the hub (so a Kaggle run can't be clobbered), and the thread keeps running across cells — no
manual 30-min re-runs needed.

Needs an HF token: if you haven't logged in yet, either set `HF_TOKEN` in section 6 below, or run
`from huggingface_hub import login; login()` in a cell (token with write permission).

In [ ]:
# ---- Auto-sync: push latest/ to HF every ~25 min (background thread) ----
import os, time, threading, tempfile, traceback, shutil
from pathlib import Path
import torch
from huggingface_hub import HfApi, hf_hub_download

CKPT_REPO = 'eulogik/TinyDoc-VLM-768-checkpoints'
OUT_DIR = Path(f'{WORK}/tinydoc-vlm/checkpoints/full768')
SYNC_CHECK_S = 120      # look for a newer step this often
SYNC_MIN_GAP_S = 1500   # min time between actual uploads (~25 min)

def _get_token():
    tok = globals().get('HF_TOKEN') or os.environ.get('HF_TOKEN', '')
    if tok and not tok.startswith('hf_xxx'):
        return tok
    try:  # Colab 'Secrets' panel (key: HF_TOKEN)
        from google.colab import userdata
        return userdata.get('HF_TOKEN') or ''
    except Exception:
        pass
    try:
        from huggingface_hub import get_token
        return get_token() or ''
    except Exception:
        return ''

def _local_step():
    f = OUT_DIR / 'latest' / 'step.txt'
    return int(f.read_text().strip()) if f.exists() else None

def _remote_step(api, token):
    try:
        p = hf_hub_download(CKPT_REPO, 'latest/step.txt', token=token)
        return int(Path(p).read_text().strip())
    except Exception:
        return -1

def _sync_now(api, token):
    step = _local_step()
    tmp = Path(tempfile.mkdtemp())
    try:
        print(f'[sync] step {step} -> fp16 (CPU), uploading ...')
        from tinydoc_vlm import TinyDocVLMForConditionalGeneration
        model = TinyDocVLMForConditionalGeneration.from_pretrained(
            str(OUT_DIR / 'latest'), trust_remote_code=True, torch_dtype=torch.float16)
        model.save_pretrained(str(tmp))
        (tmp / 'step.txt').write_text(str(step))
        api.upload_folder(folder_path=str(tmp), repo_id=CKPT_REPO, repo_type='model',
                          path_in_repo='latest', token=token,
                          commit_message=f'colab auto-sync step {step} (fp16)')
        print(f'[sync] uploaded step {step} -> {CKPT_REPO}/latest')
        return True
    except Exception:
        traceback.print_exc()
        return False
    finally:
        try:
            shutil.rmtree(str(tmp))
        except Exception:
            pass

def _loop():
    token = _get_token()
    if not token:
        print('[sync] WARNING: no HF token found -- auto-sync DISABLED')
        print('[sync]   In section 6 below, set HF_TOKEN = "hf_<your token>" or run login().')
        return
    api = HfApi()
    last_upload_step, last_upload_t = -1, 0.0
    while True:
        time.sleep(SYNC_CHECK_S)
        try:
            step = _local_step()
            if step is None or step <= last_upload_step:
                continue
            if time.time() - last_upload_t < SYNC_MIN_GAP_S:
                continue
            remote = _remote_step(api, token)
            if step <= remote:
                print(f'[sync] repo already at step {remote}; local {step} skipped')
                last_upload_step = step
                continue
            if _sync_now(api, token):
                last_upload_step, last_upload_t = step, time.time()
        except Exception:
            traceback.print_exc()

threading.Thread(target=_loop, daemon=True).start()
print(f'[sync] auto-sync thread started (upload ~every {SYNC_MIN_GAP_S//60} min, check every {SYNC_CHECK_S}s)')

In [ ]:
import os, sys, subprocess, shlex, shutil
from pathlib import Path

REPO = f'{WORK}/tinydoc-vlm'
os.chdir(REPO)
import torch
if not torch.cuda.is_available():
    raise SystemExit(
        'NO CUDA GPU: this session has no GPU (or PyTorch lacks CUDA support).\n'
        'Fix: Runtime -> Change runtime type -> GPU (T4), then Runtime -> Restart session '
        'and re-run all. Free-tier Colab gives a CPU runtime when the GPU quota is used up.')
STEPS = 30000
OUT = 'checkpoints/full768'
FINAL = Path(OUT) / 'final'

def run_streaming(cmd):
    print('$', shlex.join(cmd))
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')   # live progress
    p.wait()
    return p.returncode

if FINAL.exists():
    print('Final checkpoint already exists — skipping training.')
else:
    # Faster settings: grad-accum=1, batch=8 (fits T4 @768 with grad-ckpt+bf16).
    # max-seq-length 512 so markdown targets aren't truncated. Resume is ON by
    # default, so a Colab disconnect + re-run continues from the last step_* ckpt.
    # save-every 500 (~1.4h at 0.1 steps/s) so a shorter Colab session still captures
    # a valid checkpoint before disconnect. Atomic save + resume validation means
    # the checkpoint survives and training continues from that point next session.
    rc = run_streaming([sys.executable, 'training/full_train.py',
                        '--model-id', 'checkpoints/init_768',
                        '--manifest', 'data/training/manifest.jsonl',
                        '--steps', str(STEPS),
                        '--batch-size', '8',
                        '--grad-accum', '1',
                        '--warmup', '500',
                        '--lr', '1e-4',
                        '--max-seq-length', '512',
                        '--save-every', '500',
                        '--save-latest-every', '50',
                        '--device', 'cuda',
                        '--bf16',
                        '--grad-checkpoint',
                        '--output-dir', OUT,
                        '--max-samples', '2000000'])
    if rc != 0:
        print(f'Training failed (exit {rc}) — see log above.')

if FINAL.exists():
    dst = Path(f'{WORK}/checkpoints/full768_final')
    shutil.copytree(FINAL, dst, dirs_exist_ok=True)
    print(f'Synced final checkpoint to {dst}')
else:
    print('Final checkpoint not found — check logs above.')

## 6. (Optional) Push trained model to Hugging Face

Paste your HF write token (Settings -> Access tokens). The pushed model is a full 768
TinyDoc-VLM (loadable with from_pretrained). It is published to a NEW repo
`eulogik/TinyDoc-VLM-768` to avoid overwriting the legacy `eulogik/TinyDoc-VLM-256M`.

In [ ]:
from huggingface_hub import login, HfApi
# Paste YOUR Hugging Face write token (Settings -> Access tokens) below.
HF_TOKEN = "hf_xxx"   # <-- replace with your token (or add Colab Secret 'HF_TOKEN' and keep hf_xxx)

tok = HF_TOKEN if HF_TOKEN not in ('', 'hf_xxx') else _get_token()
if not tok:
    raise SystemExit('NO HF TOKEN: paste your write token below, or add a Colab Secret "HF_TOKEN".')
login(token=tok)
HfApi().upload_folder(
    folder_path="checkpoints/full768/final",
    repo_id="eulogik/TinyDoc-VLM-768",   # NEW repo, do not overwrite the 256M base
    repo_type="model",
    token=tok,
)
print("Pushed to Hub")